# Experiment 10 — XTR/WARP Colab workflow

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU).
2. Choose **Runtime → Run all**.
3. Upload `experiment_10_colab_input.zip` when prompted.

The notebook verifies all input checksums, creates an isolated pinned Python 3.8 environment without restarting the notebook, runs a smoke index/retrieval test, builds the complete 4-bit WARP index on GPU, freezes all 600 Top-1000 rankings in WARP's required CPU mode with `nprobe=32`, verifies the output archive, and downloads `experiment_10_colab_output.zip`.

The pretrained `google/xtr-base-en` model is used as published; no weights are fine-tuned.

Required input bundle: revision 3. The notebook rejects older ZIPs before environment creation.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, shutil, stat, subprocess, zipfile

INPUT_FILENAME = "experiment_10_colab_input.zip"
EXPECTED_INPUT_SHA256 = "511ad7b4231c7f4529590a6a8efa8ecd65d2179af96aa1c1636db2a4994fe538"
WORK_DIR = Path("/content/experiment_10_work")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_ZIP = Path("/content/experiment_10_colab_output.zip")

def _hash_stream(stream):
    digest = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
    return digest.hexdigest()

def _safe_infos(archive, *, max_uncompressed=None):
    infos = [item for item in archive.infolist() if not item.is_dir()]
    names = [item.filename for item in infos]
    if len(names) != len(set(names)):
        raise RuntimeError("Archive contains duplicate member names")
    if len(infos) > 1000:
        raise RuntimeError("Archive contains unexpectedly many members")
    total = 0
    for item in infos:
        path = PurePosixPath(item.filename)
        mode = item.external_attr >> 16
        if path.is_absolute() or not path.parts or ".." in path.parts or "" in path.parts:
            raise RuntimeError(f"Unsafe archive path: {item.filename!r}")
        if stat.S_ISLNK(mode) or item.flag_bits & 0x1:
            raise RuntimeError(f"Links/encrypted members are not allowed: {item.filename!r}")
        total += item.file_size
    if max_uncompressed is not None and total > max_uncompressed:
        raise RuntimeError(f"Input expands to {total:,} bytes, over the safety limit")
    return infos

def _verify_zip(archive, expected_type, *, max_uncompressed=None):
    infos = _safe_infos(archive, max_uncompressed=max_uncompressed)
    by_name = {item.filename: item for item in infos}
    if "manifest.json" not in by_name or "SHA256SUMS" not in by_name:
        raise RuntimeError("Archive is missing manifest.json or SHA256SUMS")
    manifest = json.load(archive.open("manifest.json"))
    if manifest.get("schema_version") != 1 or manifest.get("archive_type") != expected_type:
        raise RuntimeError("Archive manifest schema/type mismatch")
    entries = manifest.get("files", [])
    declared = {entry["path"]: entry for entry in entries}
    if len(declared) != len(entries):
        raise RuntimeError("Manifest contains duplicate paths")
    expected = set(declared) | {"manifest.json", "SHA256SUMS"}
    if set(by_name) != expected:
        raise RuntimeError(f"Archive member mismatch: missing={sorted(expected-set(by_name))}, unexpected={sorted(set(by_name)-expected)}")
    for name, entry in declared.items():
        if by_name[name].file_size != int(entry["bytes"]):
            raise RuntimeError(f"Size mismatch for {name}")
        with archive.open(name) as stream:
            if _hash_stream(stream) != entry["sha256"]:
                raise RuntimeError(f"SHA-256 mismatch for {name}")
    sums = {}
    for line in archive.read("SHA256SUMS").decode("utf-8").splitlines():
        digest, name = line.split("  ", 1)
        if name in sums or len(digest) != 64:
            raise RuntimeError("Malformed SHA256SUMS")
        sums[name] = digest
    if set(sums) != set(declared) | {"manifest.json"}:
        raise RuntimeError("SHA256SUMS coverage mismatch")
    for name, digest in sums.items():
        with archive.open(name) as stream:
            if _hash_stream(stream) != digest:
                raise RuntimeError(f"SHA256SUMS mismatch for {name}")
    return manifest, infos

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
INPUT_DIR.mkdir()
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

print(f"Upload {INPUT_FILENAME} when the picker opens...")
uploaded = files.upload()
if INPUT_FILENAME not in uploaded:
    raise RuntimeError(f"Expected {INPUT_FILENAME}; received {sorted(uploaded)}")
archive_bytes = uploaded[INPUT_FILENAME]
archive_path = WORK_DIR / INPUT_FILENAME
archive_path.write_bytes(archive_bytes)
actual_input_sha256 = hashlib.sha256(archive_bytes).hexdigest()
if actual_input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError(
        f"Outdated or mismatched input ZIP. Expected SHA-256 {EXPECTED_INPUT_SHA256}, "
        f"received {actual_input_sha256}. Upload the ZIP distributed with this notebook."
    )
del uploaded, archive_bytes

with zipfile.ZipFile(archive_path) as archive:
    input_manifest, input_infos = _verify_zip(
        archive, "experiment_10_colab_input", max_uncompressed=256 * 1024 * 1024
    )
    for item in input_infos:
        target = INPUT_DIR.joinpath(*PurePosixPath(item.filename).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(item) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=1024 * 1024)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True, capture_output=True
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError(
        "No active NVIDIA GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."
    )
print("Input verified:", input_manifest["document_count"], "documents /", input_manifest["query_count"], "queries")
print("GPU:", gpu.stdout.strip())
print("The isolated Python 3.8 environment will run as subprocesses; no notebook restart is needed.")


In [ ]:
import os, subprocess, sys, time

def run_live(command):
    print("Starting Experiment 10. Index construction is the long stage; keep this tab connected.\n")
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=str(WORK_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        process.terminate()
        raise
    status = process.wait()
    if status:
        logs = sorted((WORK_DIR / "logs").glob("*.log"), key=lambda path: path.stat().st_mtime)
        tail = ""
        if logs:
            tail = "\n".join(logs[-1].read_text(encoding="utf-8", errors="replace").splitlines()[-40:])
        raise RuntimeError(f"Experiment 10 failed with exit code {status}. Last log tail:\n{tail}")
    print(f"\nPipeline elapsed: {(time.perf_counter()-started)/60:.2f} minutes")

run_live([
    "bash",
    str(INPUT_DIR / "bootstrap_and_run.sh"),
    str(INPUT_DIR),
    str(WORK_DIR),
    str(OUTPUT_ZIP),
])


In [ ]:
if not OUTPUT_ZIP.is_file():
    raise RuntimeError("The pipeline finished without experiment_10_colab_output.zip")
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    output_manifest, _ = _verify_zip(archive, "experiment_10_colab_output")
    required = {
        "rankings/rankings.jsonl",
        "rankings/rankings.tsv",
        "rankings/query_manifest.json",
        "pid_to_asin.json",
        "provenance.json",
        "xtr_warp_compat.patch",
    }
    present = {entry["path"] for entry in output_manifest["files"]}
    missing = required - present
    if missing or not any(name.startswith("warp_index/") for name in present):
        raise RuntimeError(f"Output is incomplete: missing={sorted(missing)}")
    query_manifest = json.load(archive.open("rankings/query_manifest.json"))
    if query_manifest.get("query_count") != 600 or query_manifest.get("top_k") != 1000:
        raise RuntimeError("Frozen ranking cardinality mismatch")

with OUTPUT_ZIP.open("rb") as stream:
    output_sha256 = _hash_stream(stream)
print(f"Verified output: {OUTPUT_ZIP.stat().st_size / (1024**3):.3f} GiB")
print("SHA-256:", output_sha256)
print("Downloading experiment_10_colab_output.zip...")
files.download(str(OUTPUT_ZIP))
